<a href="https://colab.research.google.com/github/chaheti89/Dataverse/blob/main/catsdogs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
files.upload()


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"chahetijha","key":"ccb4034b531d61eafed1da8def5e647d"}'}

In [3]:
!pip install -q kaggle

In [4]:
!rm -rf ~/.kaggle

In [5]:
!mkdir -p ~/.kaggle

In [7]:
!cp /content/kaggle.json ~/.kaggle/

In [8]:
!chmod 600 ~/.kaggle/kaggle.json

In [9]:
!kaggle datasets list -s cats


ref                                                        title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
---------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
shaunthesheep/microsoft-catsvsdogs-dataset                 Cats-vs-Dogs                                         825979578  2020-03-12 05:34:30.730000          88968        837  0.875            
marquis03/cats-and-dogs                                    Cats and Dogs                                         10219362  2023-10-27 10:48:19.220000          11549        110  1.0              
chetankv/dogs-cats-images                                  Dogs & Cats Images                                   455718914  2018-04-19 18:20:08.593000          48286        611  0.5625           
waqi786/cats-dataset     

In [10]:
!kaggle datasets download -d marquis03/cats-and-dogs


Dataset URL: https://www.kaggle.com/datasets/marquis03/cats-and-dogs
License(s): apache-2.0
  0% 0.00/9.75M [00:00<?, ?B/s]
100% 9.75M/9.75M [00:00<00:00, 756MB/s]


In [11]:
!unzip -q cats-and-dogs.zip -d cats_and_dogs


In [12]:
import os
os.listdir("cats_and_dogs")


['val.csv', 'train.csv', 'train', 'val']

In [13]:
!ls cats_and_dogs
!ls cats_and_dogs/train | head
!ls cats_and_dogs/val | head


train  train.csv  val  val.csv
cat
classname.txt
dog
cat
classname.txt
dog


In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential


In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [18]:
train_dir = "cats_and_dogs/train"
val_dir = "cats_and_dogs/val"


In [19]:
# Apply data augmentation for training images
train_datagen = ImageDataGenerator(
    rescale=1./255,           # Normalize pixel values (0–1)
    rotation_range=30,        # Random rotation
    width_shift_range=0.2,    # Random horizontal shift
    height_shift_range=0.2,   # Random vertical shift
    shear_range=0.2,          # Shear transformation
    zoom_range=0.2,           # Zoom in/out
    horizontal_flip=True,     # Flip images
    fill_mode='nearest'       # Fill pixels after transformations
)

# Validation data should not be augmented — just rescaled
val_datagen = ImageDataGenerator(rescale=1./255)


In [20]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),   # Resize all images to 150x150
    batch_size=32,
    class_mode='binary'       # Since we have 2 classes: cat/dog
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary'
)


Found 275 images belonging to 2 classes.
Found 70 images belonging to 2 classes.


In [21]:
model = Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dropout(0.5),  # 💡 Prevent overfitting

    layers.Dense(512, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)



In [23]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop]
)



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - accuracy: 0.5156 - loss: 0.6924 - val_accuracy: 0.6571 - val_loss: 0.6466
Epoch 2/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.6634 - loss: 0.6337 - val_accuracy: 0.6571 - val_loss: 0.6466
Epoch 3/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.6763 - loss: 0.6060 - val_accuracy: 0.6571 - val_loss: 0.6288
Epoch 4/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 20s 2s/step - accuracy: 0.6500 - loss: 0.6150 - val_accuracy: 0.6571 - val_loss: 0.6605
Epoch 5/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.6078 - loss: 0.6468 - val_accuracy: 0.6571 - val_loss: 0.6472
Epoch 6/30
9/9 ━━━━━━━━━━━━━━━━━━━━ 19s 2s/step - accuracy: 0.6775 - loss: 0.5842 - val_accuracy: 0.6714 - val_loss: 0.6318
